# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice: Decision Tree Classifier

I will use a Decision Tree Classifier for my Search Intelligence lane. It can learn simple decision rules from features such as search impressions, clicks, and average search position. It is easy to interpret, so I can understand why the model flags certain content for review.

I will compare the model with my Week 4 rule-based baseline using the same evaluation data and metric. The goal is to check whether the model improves the ranking of content that needs attention, not just to build a more complex model.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will split the data by client so that content from the same client does not appear in both the training and evaluation sets. This helps test whether the model generalizes to clients it has not seen before, rather than memorizing client-specific patterns.

I will keep the Week 4 baseline and the Decision Tree evaluation on the same held-out clients, using the same evaluation metric. This makes the comparison fair and helps avoid an overly optimistic result.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect the February and March columns
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected successfully!")

DuckDB connected successfully!


In [17]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("February and March data paths are ready!")

February and March data paths are ready!


In [18]:

feb_monthly = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS feb_impressions,
    SUM(gsc_clicks) AS feb_clicks,
    AVG(gsc_avg_position) AS feb_avg_position
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

print("February aggregated rows:", len(feb_monthly))
display(feb_monthly.head())


February aggregated rows: 153559


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,5.500000
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,5.000000
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.262333
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.407819
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,6.961538


In [19]:

mar_monthly = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS mar_impressions,
    SUM(gsc_clicks) AS mar_clicks
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

print("March aggregated rows:", len(mar_monthly))
display(mar_monthly.head())


March aggregated rows: 176738


,client_hash_id,content_hash_id,mar_impressions,mar_clicks
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0


In [20]:

import pandas as pd

data = feb_monthly.merge(
    mar_monthly,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

data["target"] = (data["mar_clicks"] < data["feb_clicks"]).astype(int)

print("Matched pairs:", len(data))
print("Declining-click cases:", data["target"].sum())
print("Non-declining cases:", (data["target"] == 0).sum())
display(data.head())


Matched pairs: 134238
Declining-click cases: 24457
Non-declining cases: 109781


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,mar_impressions,mar_clicks,target
0,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,5.000000,3.0,0.0,1
1,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.262333,43.0,0.0,0
2,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.407819,255.0,0.0,0
3,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,6.961538,15.0,0.0,0
4,client_3ffa76342f366962,content_0674cc4ae0f68a90,74.0,1.0,8.063910,11.0,0.0,1


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error analysis and interpretation

The model may incorrectly classify content whose clicks fluctuate naturally between February and March. Content with low impressions may also have unstable click counts, making the predictions less reliable.

The model may rely heavily on February impressions, clicks, and average search position. These features can help identify patterns, but they do not explain why performance changed.

The target is based on whether March clicks are lower than February clicks. A lower click count does not necessarily mean the content is declining, because traffic can fluctuate. The model's predictions should therefore be treated as directional signals for review, not proof that content needs updating.

To improve reliability, I would inspect misclassified examples, compare the model with the Week 4 baseline using the same evaluation data and metric, and test the model on additional time periods.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.